<a href="https://www.kaggle.com/code/koushikkumardinda/predicting-student-test-scores?scriptVersionId=299210292" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

# ====================================================
# 1. Load Data
# ====================================================
# In a real Kaggle notebook, un-comment the lines below:
df_train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
df_test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')

# --- SYNTHETIC DATA GENERATION FOR REPRODUCIBILITY ---
# (I am creating dummy data so you can run this cell immediately to verify logic)
# np.random.seed(42)
# n_samples = 5000
# df_train = pd.DataFrame({
#     'id': np.arange(n_samples),
#     'study_hours': np.random.normal(10, 3, n_samples),
#     'class_attendance': np.random.randint(50, 100, n_samples),
#     'sleep_hours': np.random.normal(7, 1.5, n_samples),
#     'course': np.random.choice(['Math', 'Science', 'History', 'Art'], n_samples),
#     'gender': np.random.choice(['M', 'F'], n_samples),
#     'exam_score': np.random.normal(75, 10, n_samples) # Target
# })
# df_train['exam_score'] = df_train['exam_score'].clip(0, 100) # Clip to realistic range

# df_test = df_train.drop(columns=['exam_score']).copy()
# df_test['id'] = df_test['id'] + n_samples
# -----------------------------------------------------

print(f"Train Shape: {df_train.shape}")
print(f"Test Shape: {df_test.shape}")

# ====================================================
# 2. Minimal EDA
# ====================================================
print("\n--- Feature Identification ---")
# Identify Numerical and Categorical columns automatically
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Remove target and ID from feature lists
target_col = 'exam_score'
if target_col in num_cols: num_cols.remove(target_col)
if 'id' in num_cols: num_cols.remove('id')

print(f"Numerical Features: {num_cols}")
print(f"Categorical Features: {cat_cols}")

# Quick check of target distribution
print(f"\nTarget Mean: {df_train[target_col].mean():.4f}")
print(f"Target Std: {df_train[target_col].std():.4f}")

# ====================================================
# 3. CRUCIAL: Robust Create Folds Function
# ====================================================
def create_folds(df, n_splits=5, seed=42):
    """
    Creates Stratified Folds for regression by binning the continuous target.
    """
    df['fold'] = -1
    
    # 1. Create bins for the continuous target
    # We use a sufficient number of bins (e.g., 10-15) to approximate the distribution
    num_bins = int(1 + np.log2(len(df))) # Sturges' rule approximation or fixed 15
    
    df['target_bin'] = pd.cut(
        df['exam_score'],
        bins=15, 
        labels=False
    )
    
    # 2. Initialize StratifiedKFold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    # 3. Assign Folds
    # We stratify based on the 'target_bin' we just created
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df['target_bin'])):
        df.loc[val_idx, 'fold'] = fold_idx
        
    # 4. Cleanup
    df = df.drop(columns=['target_bin'])
    
    return df

# Apply the function
df_train = create_folds(df_train, n_splits=5)

# ====================================================
# 4. Verification
# ====================================================
print("\n--- Validation Consistency Check ---")
print("We want the Mean and Std of 'exam_score' to be nearly identical across all folds.")

stats = df_train.groupby('fold')['exam_score'].agg(['count', 'mean', 'std', 'min', 'max'])
print(stats)

# Visual check (Optional but recommended)
# The distribution of scores for every fold should overlap perfectly
plt.figure(figsize=(10, 4))
for i in range(5):
    sns.kdeplot(df_train[df_train['fold']==i]['exam_score'], label=f'Fold {i}')
plt.title('Target Distribution per Fold (Stratified)')
plt.legend()
plt.show()

print("\nReady for Feature Engineering!")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew

# ====================================================
# Feature Engineering Function
# ====================================================
def feature_engineer(df, is_train=True):
    """
    Applies feature engineering to the dataset.
    
    Args:
        df: The DataFrame (train or test)
        is_train: Boolean to indicate if it's the training set
    
    Returns:
        df: The processed DataFrame
    """
    df = df.copy()
    
    # ------------------------------------------------
    # 1. Interaction Features (Domain Knowledge)
    # ------------------------------------------------
    # Logic: High attendance AND high study hours likely compound success
    df['total_productivity'] = df['study_hours'] * df['class_attendance']
    
    # Logic: Ratio of study to sleep (balance indicator)
    # Add small epsilon to avoid division by zero
    df['study_to_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1e-5)
    
    # Logic: Interaction between course difficulty (proxy) and attendance
    # (Note: This is strictly numerical interaction; categorical interactions come later)
    
    # ------------------------------------------------
    # 2. Log Transformations (Skewness Handling)
    # ------------------------------------------------
    # Check skewness of study_hours (common in time-based data)
    # We use a threshold of 0.75.
    if abs(skew(df['study_hours'])) > 0.75:
        df['log_study_hours'] = np.log1p(df['study_hours'])
    
    # ------------------------------------------------
    # 3. Categorical Encoding Preparation
    # ------------------------------------------------
    # Identify Low vs High Cardinality
    # For this Playground, let's assume 'gender' is low, 'course' is high.
    
    # Low Cardinality -> One-Hot Encoding
    # We use pd.get_dummies. In a pipeline, usually better to align train/test cols,
    # but for this script, we'll assume consistent categories or use align later.
    low_card_cols = ['gender'] # Add others like 'internet_access' if they exist
    df = pd.get_dummies(df, columns=low_card_cols, drop_first=True, dtype=int)
    
    # High Cardinality -> Target Encoding
    # STRATEGY: We leave these columns AS IS for now.
    # We will encode them inside the Cross-Validation loop (Step 3).
    # This prevents the model from "seeing" the validation target via the encoding.
    
    return df

# ====================================================
# Apply Engineering
# ====================================================
print("--- Applying Feature Engineering ---")

# Save target and fold before processing to keep things clean
target = df_train['exam_score']
folds = df_train['fold']
df_train_feats = df_train.drop(columns=['exam_score', 'fold'])

# Apply to Train and Test
df_train_eng = feature_engineer(df_train_feats, is_train=True)
df_test_eng = feature_engineer(df_test, is_train=False)

# Reattach Target and Fold to Train (Crucial for Step 3)
df_train_eng['exam_score'] = target
df_train_eng['fold'] = folds

# ====================================================
# Quality Check
# ====================================================
print("\n--- Final Data Shapes ---")
print(f"Train Encoded Shape: {df_train_eng.shape}")
print(f"Test Encoded Shape:  {df_test_eng.shape}")

print("\n--- New Features Preview ---")
new_cols = ['total_productivity', 'study_to_sleep_ratio']
print(df_train_eng[new_cols].head(3))

print("\n--- Pending Actions ---")
print("High Cardinality columns (e.g., 'course') are still raw categorical strings.")
print("We will apply Target Encoding to these INSIDE the training loop in Step 3.")

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from category_encoders import CatBoostEncoder
import pandas as pd
import numpy as np
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ====================================================
# 1. Configuration
# ====================================================
# Identify categorical columns for CatBoost and Encoding
cat_cols = df_train_eng.select_dtypes(include=['object', 'category']).columns.tolist()
features = [col for col in df_train_eng.columns if col not in ['id', 'exam_score', 'fold']]

# Initialize Storage
oof_preds = pd.DataFrame(index=df_train_eng.index)
test_preds_accum = pd.DataFrame(index=df_test_eng.index)

# Create columns for each model
models = ['xgb', 'lgb', 'cat']
for model_name in models:
    oof_preds[model_name] = 0.0
    test_preds_accum[model_name] = 0.0

# Store scores
scores = {m: [] for m in models}

# ====================================================
# 2. Training Loop
# ====================================================
n_folds = df_train_eng['fold'].nunique()

print(f"Starting Training on {n_folds} Folds...")
print(f"Features: {features}")

for fold in range(n_folds):
    print(f"\n--- Fold {fold + 1} / {n_folds} ---")
    
    # A. Split Data
    # --------------------------------------------
    trn_idx = df_train_eng['fold'] != fold
    val_idx = df_train_eng['fold'] == fold
    
    X_train_raw = df_train_eng.loc[trn_idx, features]
    y_train = df_train_eng.loc[trn_idx, 'exam_score']
    
    X_val_raw = df_train_eng.loc[val_idx, features]
    y_val = df_train_eng.loc[val_idx, 'exam_score']
    
    X_test_raw = df_test_eng[features]
    
    # B. Dynamic Encoding (Crucial for XGB/LGBM)
    # --------------------------------------------
    # We encode INSIDE the loop to avoid leakage. 
    # CatBoostEncoder is excellent for regression.
    encoder = CatBoostEncoder(cols=cat_cols)
    
    # Fit on TRAIN, transform VAL and TEST
    X_train_enc = encoder.fit_transform(X_train_raw, y_train)
    X_val_enc = encoder.transform(X_val_raw)
    X_test_enc = encoder.transform(X_test_raw)
    
    # C. Model 1: XGBoost (Fixed for v2.0+)
    # --------------------------------------------
    xgb_model = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=42,
        early_stopping_rounds=50, # Moved to constructor
        eval_metric='rmse'        # Explicit metric
    )
    
    xgb_model.fit(
        X_train_enc, y_train,
        eval_set=[(X_val_enc, y_val)],
        verbose=False
    )
    
    val_pred_xgb = xgb_model.predict(X_val_enc)
    test_pred_xgb = xgb_model.predict(X_test_enc)
    
    oof_preds.loc[val_idx, 'xgb'] = val_pred_xgb
    test_preds_accum['xgb'] += test_pred_xgb / n_folds
    
    # Fixed RMSE calculation
    rmse_xgb = np.sqrt(mean_squared_error(y_val, val_pred_xgb))
    scores['xgb'].append(rmse_xgb)
    print(f"XGB RMSE: {rmse_xgb:.5f}")

    # D. Model 2: LightGBM
    # --------------------------------------------
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=-1, 
        num_leaves=31,
        objective='rmse',
        n_jobs=-1,
        random_state=42,
        verbosity=-1
    )
    
    lgb_model.fit(
        X_train_enc, y_train,
        eval_set=[(X_val_enc, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    val_pred_lgb = lgb_model.predict(X_val_enc)
    test_pred_lgb = lgb_model.predict(X_test_enc)
    
    oof_preds.loc[val_idx, 'lgb'] = val_pred_lgb
    test_preds_accum['lgb'] += test_pred_lgb / n_folds
    
    # Fixed RMSE calculation
    rmse_lgb = np.sqrt(mean_squared_error(y_val, val_pred_lgb))
    scores['lgb'].append(rmse_lgb)
    print(f"LGB RMSE: {rmse_lgb:.5f}")

    # E. Model 3: CatBoost
    # --------------------------------------------
    cat_model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )
    
    train_pool = Pool(X_train_raw, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val_raw, y_val, cat_features=cat_cols)
    test_pool = Pool(X_test_raw, cat_features=cat_cols)
    
    cat_model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        use_best_model=True
    )
    
    val_pred_cat = cat_model.predict(val_pool)
    test_pred_cat = cat_model.predict(test_pool)
    
    oof_preds.loc[val_idx, 'cat'] = val_pred_cat
    test_preds_accum['cat'] += test_pred_cat / n_folds
    
    # Fixed RMSE calculation
    rmse_cat = np.sqrt(mean_squared_error(y_val, val_pred_cat))
    scores['cat'].append(rmse_cat)
    print(f"CAT RMSE: {rmse_cat:.5f}")

# ====================================================
# 3. Final Evaluation & Saving
# ====================================================
print("\n--- Training Complete ---")
for m in models:
    avg_score = np.mean(scores[m])
    print(f"Average {m.upper()} RMSE: {avg_score:.5f}")

# Save OOF and Test Preds for Ensembling
oof_preds['exam_score'] = df_train_eng['exam_score'] # Add true target for regression
oof_preds.to_csv('oof_predictions.csv', index=False)
test_preds_accum.to_csv('test_predictions.csv', index=False)

print("\nFiles saved: 'oof_predictions.csv', 'test_predictions.csv'")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error

# ====================================================
# 1. Load Predictions (from Step 3)
# ====================================================
# If running sequentially, variables might still be in memory, 
# but loading ensures this step is independent.
oof_df = pd.read_csv('/kaggle/working/oof_predictions.csv')
test_preds_df = pd.read_csv('/kaggle/working/test_predictions.csv')

# Define input columns (the models) and target
model_cols = ['xgb', 'lgb', 'cat']
target_col = 'exam_score'

X_oof = oof_df[model_cols]
y_oof = oof_df[target_col]
X_test = test_preds_df[model_cols]

print(f"Ensembling {len(model_cols)} models: {model_cols}")

# ====================================================
# 2. Train Meta-Learner (Ridge Regression)
# ====================================================
# RidgeCV automatically finds the best alpha (regularization strength)
# Alphas: 0.1 (low reg), 1.0 (standard), 10.0 (high reg/conservative)
meta_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
meta_model.fit(X_oof, y_oof)

# ====================================================
# 3. Analyze Weights
# ====================================================
weights = pd.Series(meta_model.coef_, index=model_cols)
print("\n--- Ensemble Weights ---")
print(weights.sort_values(ascending=False))
print(f"Intercept: {meta_model.intercept_:.4f}")
print(f"Best Alpha: {meta_model.alpha_}")

# Check OOF Score of the Ensemble
oof_ensemble_preds = meta_model.predict(X_oof)
oof_score = np.sqrt(mean_squared_error(y_oof, oof_ensemble_preds))
print(f"\nEnsemble OOF RMSE: {oof_score:.5f}")

# Compare to single models
best_single_model = oof_df[model_cols].apply(
    lambda x: np.sqrt(mean_squared_error(y_oof, x))
).min()
print(f"Best Single Model RMSE: {best_single_model:.5f}")
print(f"Improvement: {best_single_model - oof_score:.5f}")

# ====================================================
# 4. Generate Final Predictions
# ====================================================
final_predictions = meta_model.predict(X_test)

# CLIP predictions: Test scores cannot be < 0 or > 100
final_predictions = np.clip(final_predictions, 0, 100)

# ====================================================
# 5. Create Submission File
# ====================================================
# We assume df_test is available from Step 1. 
# If not, reload it: df_test = pd.read_csv('path/to/test.csv')

submission = pd.DataFrame({
    'id': df_test['id'],
    'exam_score': final_predictions
})

# Quick Sanity Check
print("\n--- Submission Stats ---")
print(submission['exam_score'].describe())

# Save
submission.to_csv('submission.csv', index=False)
print("\n✅ Success! Saved to 'submission.csv'")